# Edge Computing Assignment 3
Hamza Khan | 22i-2149 | Section A

Actual saved experiment results from the documented Windows CPU. This notebook displays the recorded outputs and checks; it does not claim a separate Colab training run. Full scripts, trained models and CSV files are in the repository.

In [1]:
from pathlib import Path
import json, csv, subprocess
ROOT = Path.cwd()
if not (ROOT / 'results/benchmark.json').exists():
    ROOT = Path('/content/edge-assignment-3-22i-2149')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/hamzairfankhan/edge-assignment-3-22i-2149',str(ROOT)],check=True)
rows = json.loads((ROOT/'results/benchmark.json').read_text())
R = {r['model']: r for r in rows}
print('Recorded CPU benchmark | 10,000 test images per model')
print('Seed 2149 | train 45,000 | validation 5,000 | 4 CPU threads')
print('Architecture: CIFAR ResNet-18; stage widths 16/32/64/128')


Recorded CPU benchmark | 10,000 test images per model
Seed 2149 | train 45,000 | validation 5,000 | 4 CPU threads
Architecture: CIFAR ResNet-18; stage widths 16/32/64/128


## 1. Complete benchmark
All measurements below come from results/benchmark.json. Batch-1 latency: 30 warm-ups and 200 measured runs.

In [2]:
print(f"{'Model':16} {'Accuracy':>9} {'MiB':>8} {'Median ms':>11} {'p95 ms':>10}")
for r in rows:
    print(f"{r['model']:16} {r['accuracy_pct']:8.2f}% {r['checkpoint_bytes']/2**20:8.3f} {r['batch1_median_ms']:11.3f} {r['batch1_p95_ms']:10.3f}")
print('File size is deployment storage, not peak RAM. No energy test claimed.')


Model             Accuracy      MiB   Median ms     p95 ms
fp32                84.27%    2.762       0.693      0.854
ptq_int8            84.24%    0.738       4.883      5.292
qat_int8            84.52%    0.738       4.882      5.154
unstructured        84.13%    2.770       0.679      0.909
structured          82.74%    1.466       0.540      0.721
File size is deployment storage, not peak RAM. No energy test claimed.


## 2. Q1: PTQ and QAT
PTQ fuses layers and calibrates on 1,024 training images. QAT starts from FP32, inserts fake quantizers, fine-tunes for 3 epochs and converts to real INT8.

In [3]:
print('Executed PTQ conversion (ptq.py):')
print((ROOT/'ptq.py').read_text())
for name in ['fp32','ptq_int8','qat_int8']:
    r=R[name]
    print(f"{name:12}: {r['correct']}/10000 correct; {r['accuracy_pct']:.2f}%; {r['checkpoint_bytes']:,} bytes")
h=json.loads((ROOT/'results/qat_history.json').read_text())
print('QAT final validation accuracy:', round(h[-1]['validation_accuracy'],2),'%')


Executed PTQ conversion (ptq.py):
from common import *
from calibrate import calibrate
if __name__=='__main__':
    setup(); x,*_=data(); m=load_float(); m.fuse(); m.qconfig=tq.get_default_qconfig('onednn')
    tq.prepare(m,inplace=True); calibrate(m,x); tq.convert(m,inplace=True)
    torch.jit.script(m).save(str(MODELS/'ptq_int8.pt'))
    print('PTQ saved; integer convolution:',type(m.blocks[0].conv1),flush=True)

fp32        : 8427/10000 correct; 84.27%; 2,896,646 bytes
ptq_int8    : 8424/10000 correct; 84.24%; 773,408 bytes
qat_int8    : 8452/10000 correct; 84.52%; 773,408 bytes
QAT final validation accuracy: 85.48 %


## 3. Q2: iterative unstructured pruning
Global L1 pruning removes 20% of remaining weights per round, with recovery training. Masks are made permanent before saving. Dense architecture remains unchanged.

In [4]:
rounds=json.loads((ROOT/'results/pruning_iterations.json').read_text())
print('Round | zero weights / eligible weights | sparsity | validation before recovery')
for r in rounds:
    print(f"{r['iteration']:5} | {r['weight_zeros']:7,} / {r['weight_count']:,} | {100*r['sparsity']:6.2f}% | {r['validation_accuracy_before_recovery']:.2f}%")
r=R['unstructured']
print(f"Final test after recovery: {r['accuracy_pct']:.2f}%")
print(f"Permanent weight sparsity: {100*r['conv_linear_weight_sparsity']:.2f}%")
print(f"Dense MACs: {r['dense_macs']:,}; parameters: {r['parameters']:,}")
print('Expected sparsity = 1 - (0.8 ** 3) = 48.8%')
print('No sparse kernel used: zeros do not reduce dense tensor dimensions.')


Round | zero weights / eligible weights | sparsity | validation before recovery
    1 | 139,811 / 699,056 |  20.00% | 85.52%
    2 | 251,660 / 699,056 |  36.00% | 84.56%
    3 | 341,139 / 699,056 |  48.80% | 83.94%
Final test after recovery: 84.13%
Permanent weight sparsity: 48.80%
Dense MACs: 35,046,656; parameters: 701,466
Expected sparsity = 1 - (0.8 ** 3) = 48.8%
No sparse kernel used: zeros do not reduce dense tensor dimensions.


## 4. Q2: reconstructed structured model
L1 filter scores select the strongest 50% of internal channels. Conv1 outputs, BN1 state and Conv2 inputs use the same retained indices. Residual output widths stay valid.

In [5]:
c=json.loads((ROOT/'results/channel_selection.json').read_text())
print('Retained hidden widths:',c['hidden_widths'])
print(f"Validation before recovery: {c['validation_before_recovery']:.2f}%")
print(f"Validation after recovery:  {c['validation_after_recovery']:.2f}%")
for name in ['fp32','structured']:
    r=R[name]
    print(f"{name:12}: parameters {r['parameters']:,}; MACs {r['dense_macs']:,}; test {r['accuracy_pct']:.2f}%")
print('Conv2 input width is reduced; Conv2 output width is preserved.')
print('Physical smaller tensors, not masked zero-valued channels.')


Retained hidden widths: [8, 8, 16, 16, 32, 32, 64, 64]
Validation before recovery: 27.36%
Validation after recovery:  83.64%
fp32        : parameters 701,466; MACs 35,046,656; test 84.27%
structured  : parameters 357,690; MACs 17,941,760; test 82.74%
Conv2 input width is reduced; Conv2 output width is preserved.
Physical smaller tensors, not masked zero-valued channels.


## 5. ONNX deployment and verification
All five exports were checked and evaluated on the full 10,000-image test set using ONNX Runtime CPU.

In [6]:
v=json.loads((ROOT/'results/onnx_validation.json').read_text())
print(f"{'Model':16} {'Accuracy':>9} {'Max error':>12} {'Q/DQ nodes':>12}")
for r in v:
    counts=f"{r['quantize_linear_nodes']}/{r['dequantize_linear_nodes']}"
    print(f"{r['model']:16} {r['accuracy_pct']:8.2f}% {r['max_absolute_logit_difference']:12.6f} {counts:>12}")
checks=json.loads((ROOT/'results/verification.json').read_text())
print('Artifact verification:',checks['status'])
for item in checks['checks']: print('PASS:',item)


Model             Accuracy    Max error   Q/DQ nodes
fp32                84.27%     0.000010          0/0
ptq_int8            84.29%     0.304893        32/82
qat_int8            84.43%     0.426419        32/82
unstructured        84.13%     0.000008          0/0
structured          82.74%     0.000010          0/0
Artifact verification: PASS
PASS: fp32: checkpoint reload, output shape, prediction counts and ONNX check passed
PASS: ptq_int8: checkpoint reload, output shape, prediction counts and ONNX check passed
PASS: qat_int8: checkpoint reload, output shape, prediction counts and ONNX check passed
PASS: unstructured: checkpoint reload, output shape, prediction counts and ONNX check passed
PASS: structured: checkpoint reload, output shape, prediction counts and ONNX check passed
PASS: 48.8% permanent unstructured weight sparsity
PASS: Structured residual dimensions preserved; parameters physically reduced
PASS: PTQ and QAT contain integer convolution operators
